# Aula 4, IA estatística

Notebook executável que acompanha a aula [04-ia-estatistica.md](../../lessons/modulo-01-introducao-ia/04-ia-estatistica.md).

## O que vamos fazer aqui

Vamos construir, do zero, um classificador de Naive Bayes que separa mensagens de
alunos em duas categorias, dúvida de conteúdo e problema técnico. Em vez de regras,
o modelo aprende contando palavras nos exemplos rotulados.

No projeto, colocamos esse classificador estatístico contra um classificador de
regras na mesma tarefa, para ver na prática quem generaliza melhor para frases novas.
Tudo é Python puro, com `math` e `collections`.

## Dados e treino do Naive Bayes

O treino aqui é só contar: quantas mensagens há de cada classe e quantas vezes cada
palavra aparece em cada classe. Essas contagens viram as probabilidades que o modelo
usa para decidir.

In [ ]:
import math
from collections import defaultdict

# Pequeno conjunto de mensagens de alunos, cada uma com a sua classe.
dados_treino = [
    ('não entendi a derivada', 'conteúdo'),
    ('como resolvo essa integral', 'conteúdo'),
    ('qual a fórmula do limite', 'conteúdo'),
    ('não consigo entender a matriz', 'conteúdo'),
    ('o vídeo não carrega', 'técnico'),
    ('a página está travando', 'técnico'),
    ('não consigo fazer login', 'técnico'),
    ('o site fica fora do ar', 'técnico'),
]


def tokenizar(texto):
    """Quebra o texto em palavras simples, tudo em minúsculas."""
    return texto.lower().split()


def treinar_naive_bayes(dados):
    """Conta classes e palavras para estimar as probabilidades depois."""
    contagem_classe = defaultdict(int)
    contagem_palavra = defaultdict(lambda: defaultdict(int))
    vocabulario = set()
    for texto, classe in dados:
        contagem_classe[classe] += 1
        for palavra in tokenizar(texto):
            contagem_palavra[classe][palavra] += 1
            vocabulario.add(palavra)
    return contagem_classe, contagem_palavra, vocabulario

## Previsão pela soma dos logaritmos

Para classificar, somamos o logaritmo do prior da classe com o logaritmo da
probabilidade de cada palavra naquela classe, usando suavização de Laplace para não
zerar palavras nunca vistas. A classe de maior soma vence. Trabalhamos com
logaritmos para evitar números minúsculos.

In [ ]:
def prever(texto, modelo):
    """Escolhe a classe mais provável usando a soma dos logaritmos."""
    contagem_classe, contagem_palavra, vocabulario = modelo
    total_docs = sum(contagem_classe.values())
    melhor_classe, melhor_score = None, float('-inf')
    for classe in contagem_classe:
        # Logaritmo do prior P(classe).
        score = math.log(contagem_classe[classe] / total_docs)
        total_palavras = sum(contagem_palavra[classe].values())
        for palavra in tokenizar(texto):
            freq = contagem_palavra[classe][palavra]
            # Verossimilhança P(palavra | classe) com suavização de Laplace.
            prob = (freq + 1) / (total_palavras + len(vocabulario))
            score += math.log(prob)
        if score > melhor_score:
            melhor_score, melhor_classe = score, classe
    return melhor_classe


modelo = treinar_naive_bayes(dados_treino)

# Mensagens novas, que não estão no treino.
for mensagem in ['não entendi o limite', 'o login não funciona', 'a matriz travou']:
    print(repr(mensagem), '->', prever(mensagem, modelo))

O modelo nunca viu essas frases exatas, mas classifica bem, porque aprendeu
que certas palavras puxam para cada classe. A frase proposital `a matriz travou`
mistura uma palavra de conteúdo (matriz) com uma de problema técnico (travou), e é
interessante observar para que lado o modelo pende e por quê.

## Projeto da aula, regras contra estatística

Vamos comparar o Naive Bayes com um classificador simples de regras, na mesma tarefa,
sobre mensagens de teste que nenhum dos dois viu. A ideia é medir quem generaliza
melhor para frases com palavras novas.

In [ ]:
REGRAS = {
    'derivada': 'conteúdo', 'integral': 'conteúdo', 'limite': 'conteúdo',
    'matriz': 'conteúdo', 'fórmula': 'conteúdo',
    'login': 'técnico', 'vídeo': 'técnico', 'página': 'técnico',
    'site': 'técnico', 'travando': 'técnico',
}


def prever_por_regras(texto):
    for palavra in tokenizar(texto):
        if palavra in REGRAS:
            return REGRAS[palavra]
    return 'desconhecido'


# Exemplos de teste, com a resposta correta, usando palavras e formas novas.
teste = [
    ('não entendi o limite', 'conteúdo'),
    ('o login não funciona', 'técnico'),
    ('a matriz não fecha', 'conteúdo'),
    ('o aplicativo congelou', 'técnico'),
    ('como calculo a integral', 'conteúdo'),
]

acertos_nb = 0
acertos_regras = 0
print(f"{'mensagem':30} {'correto':10} {'naive bayes':12} {'regras':10}")
for texto, correto in teste:
    p_nb = prever(texto, modelo)
    p_re = prever_por_regras(texto)
    acertos_nb += (p_nb == correto)
    acertos_regras += (p_re == correto)
    print(f'{texto:30} {correto:10} {p_nb:12} {p_re:10}')

print()
print('Acertos Naive Bayes:', acertos_nb, 'de', len(teste))
print('Acertos por regras :', acertos_regras, 'de', len(teste))

Em geral o Naive Bayes acerta mais, porque combina evidências de várias
palavras, enquanto o método de regras falha quando a palavra-chave esperada não
aparece, como em `o aplicativo congelou`. Para o projeto, escreva um parágrafo
comparando as duas abordagens, ligando com o que você viu nas aulas de IA simbólica e
estatística.